Ссылка на урок: [Спринт 2/5: 2 RAG: от сырых данных до точных ответов → Тема 3/4: Архитектура RAG → Урок 2/6: Retrieval-пайплайн: от запроса к релевантным чанкам]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/b36d9f5f-3f26-4fbb-bd65-71d33c4523ef/lessons/accc059f-d252-4948-a562-afd29ac0e7f2/#8b62d47b-1ccf-42cd-bfe0-248c956620fc )

In [1]:
import json

with open("corpus.json", encoding="utf-8") as f:
    data = json.load(f)

chunks = data["chunks"]      # list[dict]: 30 чанков, поля id, category, text
queries = data["queries"]    # list[dict]: 10 запросов, поля query и relevant_ids

print("Чанков:", len(chunks), "| Запросов:", len(queries))
print("Пример чанка:", chunks[0])
print("Пример запроса:", queries[6])

Чанков: 30 | Запросов: 10
Пример чанка: {'id': 0, 'category': 'поддержка', 'text': 'Чтобы сбросить пароль, на странице входа нажмите «Забыли пароль», введите электронную почту и перейдите по ссылке из письма.'}
Пример запроса: {'query': 'Какой штраф для ИП на УСН за просрочку налоговой декларации?', 'relevant_ids': [8]}


In [5]:
import json
import chromadb
from sentence_transformers import SentenceTransformer

# --- Загружаем корпус: 30 чанков базы знаний и 10 тестовых запросов ---
with open("corpus.json", encoding="utf-8") as f:
    data = json.load(f)
chunks = data["chunks"]
queries = data["queries"]
texts = [c["text"] for c in chunks]
metas = [{"category": c["category"]} for c in chunks]

# --- Эмбеддер: тот же multilingual-e5, что и в прошлых уроках (на CPU) ---
embedder = SentenceTransformer("intfloat/multilingual-e5-base")

# Чанки кодируем заранее с префиксом passage: — это предвычисление документов
doc_vecs = embedder.encode(["passage: " + t for t in texts],
                           normalize_embeddings=True, show_progress_bar=False)

# --- Кладём чанки в локальную векторную базу Chroma ---
client = chromadb.EphemeralClient()             # База в оперативной памяти
try:
    client.delete_collection("knowledge_base")              # Чистим прошлый прогон
except Exception:
    pass
col = client.create_collection(
    name="knowledge_base",
    configuration={"hnsw": {"space": "cosine"}}, # Косинусная близость
)
col.add(
    ids=[str(c["id"]) for c in chunks],         # id чанка строкой
    embeddings=doc_vecs.tolist(),               # Готовые векторы чанков
    documents=texts,                            # Сами тексты
    metadatas=metas,                            # Метаданные (категория)
)


def retrieve(query, col, embedder, k=5, threshold=0.0):
    """Retrieval-пайплайн: embed query -> search -> filter -> return.

    query — текст запроса; k — сколько чанков вернуть;
    threshold — минимальное косинусное сходство, ниже него чанк не берём.
    Возвращает список (chunk_id, text, similarity) по убыванию сходства.
    """
    # 1. embed query: кодируем запрос той же моделью, префикс query:
    q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
    # 2. search: ищем k ближайших по косинусу в векторной базе
    # Пропуск 1: запросите топ-k к col по вектору q_vec
    res = col.query(
        query_embeddings=[q_vec.tolist()],
        n_results=k,
    )
    # 3. filter: переводим расстояние Chroma в сходство и отсекаем по порогу
    out = []
    for cid, doc, dist in zip(
            res["ids"][0],
            res["documents"][0],
            res["distances"][0],
    ):
        sim = 1.0 - dist                        # Chroma отдаёт расстояние 1 - cos
        # Пропуск 2: добавьте (cid, doc, sim) в out, только если sim >= threshold
        if  sim >= threshold:
            out.append((
                cid,
                doc,
                sim,
            ))
    # 4. return: top-k уже отсортированы базой по убыванию сходства
    return out


# --- Прогоняем 10 тестовых запросов через пайплайн ---
THRESHOLD = 0.83
for qi, q in enumerate(queries):
    # Пропуск 3: получите топ-3 чанка для q["query"] с порогом THRESHOLD
    hits = retrieve(q["query"], col, embedder, k=3, threshold=THRESHOLD)
    print(f"\n[{qi}] {q['query']}")
    if not hits:
        print("    (ничего выше порога — пайплайн вернул пусто)")
    for cid, doc, sim in hits:
        print(f"    sim={sim:.3f} #{cid} {doc[:58]}")


[0] Как сбросить пароль в личном кабинете?
    sim=0.884 #0 Чтобы сбросить пароль, на странице входа нажмите «Забыли п

[1] Сколько времени занимает доставка по Москве?
    sim=0.908 #1 Доставка курьером по Москве занимает один-два рабочих дня 

[2] Какая гарантия на крупную бытовую технику?
    sim=0.896 #2 На крупную бытовую технику действует гарантия производител

[3] Как связаться с технической поддержкой?
    sim=0.878 #3 Техническая поддержка отвечает по телефону 8-800-555-35-35

[4] Можно ли оплатить заказ картой при получении?
    sim=0.885 #4 Оплата картой возможна при получении заказа курьеру или в 
    sim=0.846 #27 Изменить способ оплаты после оформления заказа нельзя — по
    sim=0.831 #18 Возврат денежных средств за оплаченный картой заказ происх

[5] Какой режим работы колл-центра?
    sim=0.875 #5 Колл-центр работает ежедневно с 8:00 до 22:00 по московско

[6] Какой штраф для ИП на УСН за просрочку налоговой декларации?
    sim=0.892 #7 Штраф за просрочку сдачи налогов

In [ ]:
# col, embedder, retrieve(), queries, chunks — из задания 4

# Пропуск 1: соберите relevant = {индекс запроса: множество relevant_ids}
relevant = <...>

problem = []
for qi, q in enumerate(queries):
    hits = retrieve(q["query"], col, embedder, k=5, threshold=0.0)  # сырой топ-5
    top_ids = [int(cid) for cid, _, _ in hits]
    rel = relevant[qi]
    # Пропуск 2: запрос проблемный, если верхний чанк не релевантен
    #            (для запроса вне базы rel пустой — верхний не может быть релевантным)
    is_problem = <...>
    if is_problem:
        problem.append(qi)

print("Проблемные запросы:", problem)

for qi in problem:
    q = queries[qi]
    rel = relevant[qi]
    hits = retrieve(q["query"], col, embedder, k=3, threshold=0.0)
    print(f"\n[{qi}] {q['query']}  (релевантные id: {sorted(rel) or 'нет'})")
    for cid, doc, sim in hits:
        # Пропуск 3: '+' если чанк релевантен (int(cid) в rel), иначе пробел
        mark = <...>
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}")

# Контраст по порогу на запросе вне базы [8]: 0.0 против 0.83
print("\nЗапрос [8] без порога:",
      [f"#{c}" for c, _, _ in retrieve(queries[8]['query'], col, embedder,
                                       k=3, threshold=0.0)])
print("Запрос [8] с порогом 0.83:",
      [f"#{c}" for c, _, _ in retrieve(queries[8]['query'], col, embedder,
                                       k=3, threshold=0.83)])

In [11]:
# col, embedder, retrieve(), queries, chunks — из задания 4

# Пропуск 1: соберите relevant = {индекс запроса: множество relevant_ids}
relevant = {i: set(querie["relevant_ids"]) for i, querie in enumerate(queries)}

problem = []
for qi, q in enumerate(queries):
    hits = retrieve(q["query"], col, embedder, k=5, threshold=0.0)  # сырой топ-5
    top_ids = [int(cid) for cid, _, _ in hits]
    rel = relevant[qi]
    # Пропуск 2: запрос проблемный, если верхний чанк не релевантен
    #            (для запроса вне базы rel пустой — верхний не может быть релевантным)
    is_problem = top_ids[0] not in rel if top_ids else False
    if is_problem:
        problem.append(qi)

print("Проблемные запросы:", problem)

for qi in problem:
    q = queries[qi]
    rel = relevant[qi]
    hits = retrieve(q["query"], col, embedder, k=3, threshold=0.0)
    print(f"\n[{qi}] {q['query']}  (релевантные id: {sorted(rel) or 'нет'})")
    for cid, doc, sim in hits:
        # Пропуск 3: '+' если чанк релевантен (int(cid) в rel), иначе пробел
        mark = "+" if int(cid) in rel else " "
        print(f"    {mark} sim={sim:.3f} #{cid} {doc[:60]}")

# Контраст по порогу на запросе вне базы [8]: 0.0 против 0.83
print("\nЗапрос [8] без порога:",
      [f"#{c}" for c, _, _ in retrieve(queries[8]['query'], col, embedder,
                                       k=3, threshold=0.0)])
print("Запрос [8] с порогом 0.83:",
      [f"#{c}" for c, _, _ in retrieve(queries[8]['query'], col, embedder,
                                       k=3, threshold=0.83)])

Проблемные запросы: [6, 7, 8]

[6] Какой штраф для ИП на УСН за просрочку налоговой декларации?  (релевантные id: [8])
      sim=0.892 #7 Штраф за просрочку сдачи налоговой декларации для организаци
    + sim=0.881 #8 Для индивидуального предпринимателя на упрощёнке штраф за пр
      sim=0.857 #22 Налоговую декларацию по УСН индивидуальный предприниматель п

[7] Сколько дней даётся на возврат товара надлежащего качества?  (релевантные id: [10])
      sim=0.865 #9 Покупатель вправе вернуть товар надлежащего качества, если о
      sim=0.845 #25 Оформить возврат товара можно в течение гарантийного срока, 
      sim=0.844 #18 Возврат денежных средств за оплаченный картой заказ происход

[8] Как оформить отпуск по уходу за ребёнком?  (релевантные id: нет)
      sim=0.798 #11 Промокод на скидку вводится в корзине в поле «Промокод» пере
      sim=0.791 #0 Чтобы сбросить пароль, на странице входа нажмите «Забыли пар
      sim=0.787 #27 Изменить способ оплаты после оформления заказа нельзя — по